# Solution: генерация ответов для сдачи

Ноутбук загружает обученную модель и создаёт `answers.jsonl` для бота. На каждую из **1919 задач** генерируется одна программа. Каждая строка результата имеет единственное поле `code`: `{"code": "<вывод модели>"}`.

Все ответы должны генерироваться не дольше **180 секунд на одной NVIDIA T4**. Здесь замеряется вызов `generate_code`, включая токенизацию и декодирование. Загрузка модели, скачивание теста и запись файла выполняются отдельно от замера.

## Пути и параметры

Надо указать **три папки в первой ячейке**:

- `RUN_PATH`: папка эксперимента с `config.yaml`, `init_state/tokenizer.json` и `checkpoints/best.pt`.
- `PROJECT_PATH`: папка репозитория с `lib/inference.py`.
- `DATA_PATH`: папка с `test.parquet`.

Если локального `test.parquet` нет, скачивается только тест с Hugging Face. Для установки и скачивания нужен интернет. Результат: `/kaggle/working/answers.jsonl` на Kaggle или `answers.jsonl` в текущей папке на локально.

In [1]:
from pathlib import Path

RUN_PATH = Path("/kaggle/input/datasets/wwip62/output-baseline-67-16h/experiment_runs/mha_6enc_7dec")
PROJECT_PATH = Path("/kaggle/input/datasets/wwip62/nlp-hw2-codegen")
DATA_PATH = Path("/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384")

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_PATH = WORK_DIR / "answers.jsonl"


RUN_PATH = RUN_PATH.expanduser().resolve()
PROJECT_PATH = PROJECT_PATH.expanduser().resolve()
DATA_PATH = DATA_PATH.expanduser().resolve()
WORK_DIR = WORK_DIR.expanduser().resolve()
OUTPUT_PATH = OUTPUT_PATH.expanduser().resolve()

In [2]:
for name, path in (("RUN_PATH", RUN_PATH), ("PROJECT_PATH", PROJECT_PATH), ("DATA_PATH", DATA_PATH)):
    if not path.is_dir():
        raise FileNotFoundError(f"{name}: папка не найдена: {path}")

required_files = [
    PROJECT_PATH / "lib" / "inference.py",
    RUN_PATH / "config.yaml",
    RUN_PATH / "init_state" / "tokenizer.json",
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Проверь пути в первой ячейке. Не найдены файлы:\n" + "\n".join(missing_files))

In [3]:
print("Репозиторий:", PROJECT_PATH)
print("Эксперимент:", RUN_PATH)
print("Данные:", DATA_PATH)
print("Результат:", OUTPUT_PATH)

Репозиторий: /kaggle/input/datasets/wwip62/nlp-hw2-codegen
Эксперимент: /kaggle/input/datasets/wwip62/output-baseline-67-16h/experiment_runs/mha_6enc_7dec
Данные: /kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384
Результат: /kaggle/working/answers.jsonl


In [4]:
%pip install -q datasets tokenizers tqdm PyYAML "sacrebleu>=2.4,<3" wandb

import importlib
import importlib.util
import json
import os
import subprocess
import sys
from time import perf_counter

os.environ["PROJECT_DIR"] = str(PROJECT_PATH)
sys.path.insert(0, str(PROJECT_PATH))
for name in list(sys.modules):
    if name == "lib" or name.startswith("lib."):
        del sys.modules[name]

Note: you may need to restart the kernel to use updated packages.


In [5]:
CHECKPOINT = "last"
DEVICE = "auto"
MAX_TOKENS = 8196
MAX_NEW_TOKENS = 128
BEAM_TOP_K = 14
VOTING_MODE = "weighted_counter"
TIME_LIMIT_SECONDS = 180
EXPECTED_QUESTIONS = 1919

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_PATH = WORK_DIR / "answers.jsonl"

## Импорты

In [6]:
import pyarrow.parquet as pq
import torch
from huggingface_hub import hf_hub_download
from IPython.display import FileLink, display

from lib.inference import generate_code, load_for_inference

## Загрузка модели и токенайзера

`load_for_inference` восстанавливает архитектуру из `config.yaml`,
загружает веса выбранного чекпойнта и переводит модель в режим `eval`.

In [7]:
model, tokenizer = load_for_inference(
    RUN_PATH, checkpoint=CHECKPOINT, device=DEVICE,
)
device = next(model.parameters()).device
print(f"Модель: {type(model).__name__} | устройство: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(device)}")

Модель: BaselinePlusPlusModel | устройство: cuda:0
GPU: Tesla T4


## Чтение тестовых вопросов

Сначала проверяем `QUESTIONS_PATH`. Если файла от бота нет, используем
`TEST_PATH`, при необходимости скачав только тестовый Parquet с Hugging Face.
Из Parquet читается только колонка вопросов.

Не сортируем, не фильтруем и не удаляем дубликаты. Некорректная строка
JSONL вызывает ошибку, чтобы позиции ответов не сдвинулись.

In [8]:
TEST_PATH = DATA_PATH / "test.parquet"

if not TEST_PATH.is_file():
    TEST_PATH = hf_hub_download(
        repo_id=DATASET_NAME,
        repo_type="dataset",
        filename="data/test/test.parquet",
    )
    
questions = pq.read_table(
    TEST_PATH, columns=["question"],
).column("question").to_pylist()
question_source = str(TEST_PATH)


print(f"Источник: {question_source}")
print(f"Прочитано вопросов: {len(questions)}")

Источник: /kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/test.parquet
Прочитано вопросов: 1919


## Генерация и замер времени

На CUDA используем FP16 autocast. Внутри `generate_code` вопросы группируются по длине, а ответы возвращаются в исходном порядке.

In [9]:
if device.type == "cuda":
    torch.cuda.synchronize(device)
started = perf_counter()
generated_codes = generate_code(
    model,
    tokenizer,
    questions,
    max_tokens=MAX_TOKENS,
    max_new_tokens=MAX_NEW_TOKENS,
    beam_top_k=BEAM_TOP_K,
    voting_mode=VOTING_MODE,
)
if device.type == "cuda":
    torch.cuda.synchronize(device)
generation_seconds = perf_counter() - started


print(f"Сгенерировано программ: {len(generated_codes)}")
print(f"Время генерации: {generation_seconds:.2f} с")

status = "выполнен" if generation_seconds <= TIME_LIMIT_SECONDS else "превышен"
print(f"Лимит {TIME_LIMIT_SECONDS} с на T4: {status}.")

Сгенерировано программ: 1919
Время генерации: 176.02 с
Лимит 180 с на T4: выполнен.


На всякий случай проверим:

In [10]:
assert len(questions) == len(generated_codes) == EXPECTED_QUESTIONS

## Сохранение в JSONL

Сохраняем вывод модели без изменения пробелов и переносов строк. `json.dumps` экранирует переносы внутри программы, поэтому каждый
JSON-объект занимает ровно одну строку файла.

In [11]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    for generated_code in generated_codes:
        file.write(json.dumps({"code": generated_code}, ensure_ascii=False) + "\n")

print("Файл:", OUTPUT_PATH)
print("Строк:", len(generated_codes))
print("Пустых ответов:", sum(not generated_code.strip() for generated_code in generated_codes))
display(FileLink(os.path.relpath(OUTPUT_PATH, Path.cwd())))

Файл: /kaggle/working/answers.jsonl
Строк: 1919
Пустых ответов: 0


/kaggle/working/answers.jsonl